# Imports

In [ ]:
# ---- Device: must run before anything imports jax ----
from devices import use_device
use_device("auto")      # "auto" | "gpu" | "cpu"  (restart the kernel to switch)

import os
import jax, numpyro
import jax.numpy as jnp
import numpy as np
print("JAX:", jax.__version__)
print(jax.default_backend())   # 'gpu' or 'cpu': where JAX runs things by default
print(jax.devices())           # e.g. [CudaDevice(id=0)] for the GPU, or 8 CpuDevices
print("local device count:", jax.local_device_count())

import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, init_to_value

from likelihood import (
    FD_gaussianlikelihood,
    FD_marg_fullnormal_gaussianlikelihood,
    draw_fluences_fullnormal_gaussmarg,
    FD_marg_halfnormal_nooverlap_gaussianlikelihood,
    draw_fluences_halfnormal_nooverlap_gaussmarg,
    wraparound_ratio_posthoc,
)
from model import (
    setup_model,
    make_orthogonal_poly_log_sed,
)
from data import (
    prepare_burst,
)
from run import (
    set_priors,
    run_fit,
    posthoc_samples,
)
from results import (
    save_run,
    to_idata,
)

from plotting import (
    plot_burst_TD, 
    plot_burst_FD,
    model_from_draw,
    plot_fit_TD,
    plot_fit_FD,
)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker
import arviz as az
import arviz_plots as azp
az.rcParams["plot.max_subplots"] = 200

import random

FRBayes: running on the GPU


/arc/home/hsiegel/FRBayes/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


JAX: 0.11.2
gpu
[CudaDevice(id=0)]
local device count: 1


# Setup

In [ ]:
from pathlib import Path

PROJECT  = Path("/arc/home/hsiegel/FRBayes")

# event_id = "22310619_BB" #Multi-pulse, 3 pulses
# data_path = PROJECT / "data" / f"event_{event_id}.npz"
# "22310619_BB"
# burst = prepare_burst(
#     path=data_path,
#     n_avg=16,                # frequency channels averaged per block
#     fd_downsample=8,        # downsampling factor of high frequency FD bins
#     toa=None,               # if no upstream TOA, falls back to noise_fraction below to estimate noise
#     noise_fraction=0.2,     # fraction of the segment used for the off-pulse noise estimate
#     verbose=True,
# )

# event_id = "27687783_BB" #Multi-pulse, many
# data_path = PROJECT / "data" / f"event_{event_id}.npz"
# burst = prepare_burst(
#     path=data_path,
#     n_avg=8,                # frequency channels averaged per block
#     fd_downsample=64,        # downsampling factor of high frequency FD bins
#     toa=None,               # if no upstream TOA, falls back to noise_fraction below to estimate noise
#     noise_fraction=0.2,     # fraction of the segment used for the off-pulse noise estimate
#     verbose=True,
# )

# event_id = "23438043_BB" # Single-pulse, Narrow SED with scatter
# data_path = PROJECT / "data" / f"event_{event_id}.npz"
# burst = prepare_burst(
#     path=data_path,
#     n_avg=16,                # frequency channels averaged per block
#     fd_downsample=8,        # downsampling factor of high frequency FD bins
#     toa=None,               # if no upstream TOA, falls back to noise_fraction below to estimate noise
#     noise_fraction=0.2,     # fraction of the segment used for the off-pulse noise estimate
#     verbose=True,
# )

event_id = "22154974_BB" # Single-pulse, Broadband with scatter
data_path = PROJECT / "data" / f"event_{event_id}.npz"
burst = prepare_burst(
    path=data_path,
    n_avg=16,                # frequency channels averaged per block
    fd_downsample=8,        # downsampling factor of high frequency FD bins
    toa=None,               # if no upstream TOA, falls back to noise_fraction below to estimate noise
    noise_fraction=0.2,     # fraction of the segment used for the off-pulse noise estimate
    verbose=True,
)

out_path  = PROJECT / "runs" / f"event_{event_id}.nc"
out_path.parent.mkdir(parents=True, exist_ok=True)     # create runs/ if it doesn't exist

In [ ]:
burst.data_fd.shape

In [ ]:
# M_FD = jnp.sum(jax.vmap(single_pulse_fn)(params), axis=0)    # (n_chan, n_fourier)

plot_burst_TD(burst);                                          # data, in time
# plot_burst_TD(burst, M_FD, title="Model")                     # model, in time
plot_burst_FD(burst);                                          # data, in Fourier space
# plot_burst_FD(burst, burst.data_fd - M_FD, title="Residual")  # residual, in Fourier space

# Run

In [ ]:
constants_dict = {
    "freq_channels_ref": burst.freq_channels_ref,
    "DM_incoherent": 0.0,          # data are coherently dedispersed, so no intra-channel smearing remains
    "channel_width": burst.channel_width,
}

single_pulse_fn = setup_model(
    constants_dict=constants_dict,
    freqs_channels=burst.freq_channels,
    times=burst.times,
    sed_fn=make_orthogonal_poly_log_sed(*burst.spectral_basis),
    fourier_freqs=burst.fourier_freqs,
)

# n_pulses = 5
# num_chains = 32
n_pulses = 5
num_chains = 8


target_accept_prob = 0.8
seed = 1
num_warmup = 500
num_samples = 200
progress_bar = True
# repulsion_dict = {"x_min":0.75}
repulsion_dict={}


priors = set_priors(burst, single_pulse_fn, n_pulses, s_opt=100.0)
# priors = set_priors(burst, single_pulse_fn, n_pulses, s_opt=100.0, gamma_range=(-10.0, 30.0), beta_range=(-10.0, 30.0))


extend_duration = False # Padding against wraparound: off by default. Turn on when a pulse reaches the window edge.
fit_fn = single_pulse_fn.extend_model_duration(burst, priors) if extend_duration else single_pulse_fn

mcmc = run_fit(burst, fit_fn, priors, FD_gaussianlikelihood, seed=seed,
               num_chains=num_chains, n_pulses=n_pulses, target_accept_prob=target_accept_prob,
               num_warmup=num_warmup, num_samples=num_samples, progress_bar=progress_bar,
               repulsion=repulsion_dict)
samples = mcmc.get_samples()

mcmc_data = to_idata(mcmc, log_likelihood={}, posterior_predictive={})
# idata = save_run(None, mcmc, burst, priors, data_path,
#                  single_pulse_fn=fit_fn, extra_samples=extra, event_id=event_id)

In [ ]:
div = np.asarray(mcmc.get_extra_fields(group_by_chain=True)["diverging"]).astype(bool)   # (chain, draw)
print("divergences per chain:", div.sum(axis=1))
# s = {k: np.asarray(v) for k, v in mcmc.get_samples(group_by_chain=True).items()}
# snr_on = 3.0                                                       # whatever you used
# checks = {
#     "closest-pair x":         s["toa_separation"].min(axis=-1),
#     "smallest A_snr":         s["A_snr"].min(axis=-1),
#     "pulse near snr_on (frac)": (np.abs(s["A_snr"] - snr_on) < 1.0).any(axis=-1).astype(float),
# }
# for name, v in checks.items():
#     print(f"{name:25s} divergent: {np.median(v[div]) if 'frac' not in name else v[div].mean():.2f}   "
#           f"others: {np.median(v[~div]) if 'frac' not in name else v[~div].mean():.2f}")

In [ ]:
steps = np.asarray(mcmc.get_extra_fields(group_by_chain=True)["num_steps"])     # (chain, draw), after warmup
depth = np.ceil(np.log2(steps + 1)).astype(int)
max_depth = 10                                                                    # whatever the run used
for c in range(steps.shape[0]):
    print(f"chain {c}: median depth {np.median(depth[c]):.0f}   max {depth[c].max()}   "
          f"hit the limit {np.mean(steps[c] >= 2 ** max_depth - 1):.0%}   mean steps {steps[c].mean():.0f}")

In [ ]:
az.summary(mcmc_data, round_to=3)

In [ ]:
axes = azp.plot_dist(mcmc_data, stats=None)
for ax in plt.gcf().axes:
    ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=4))  # Limit to 4 ticks max
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.subplots_adjust(wspace=0.1)
plt.subplots_adjust(hspace=2.5)

# Remove all "mean" text annotations across all subplots
for ax in plt.gcf().axes:
    for text in list(ax.texts):
        if "mean" in text.get_text():
            text.remove()
            # Remove horizontal interval lines (they only have 2 x-data points)
    for line in list(ax.lines):
        if len(line.get_xdata()) <= 2:
            line.remove()
            # 3. Remove point estimate dots
    for collection in list(ax.collections):
        collection.remove()

plt.draw() 

In [ ]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="Explicitly requested dtype.*is not available",
    category=UserWarning,
)

azp.plot_trace(mcmc_data)

# 2. Clean up x-axis tick clutter across all generated subplots
for ax in plt.gcf().axes:
    ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=4))
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
pe = np.asarray(mcmc.get_extra_fields(group_by_chain=True)["potential_energy"])
s = {k: np.asarray(v) for k, v in mcmc.get_samples(group_by_chain=True).items()}
active = s["A_snr"] > 3.0
for c in range(pe.shape[0]):
    t = np.nanmedian(np.where(active[c], s["mean_arrival_time"][c], np.nan), axis=0)
    print(f"chain {c}: -log post {pe[c].mean():9.1f} ± {pe[c].std():4.1f}   "
          f"active {active[c].sum(-1).mean():.1f}   t [ms] {np.round(np.sort(t) * 1e3, 2)}   "
          f"A_snr {np.round(np.median(s['A_snr'][c], axis=0), 1)}")

In [ ]:
key = jax.random.PRNGKey(31)

In [ ]:
# -------------------------------------------------------------------
# 1. Extract a Random Posterior Draw & Evaluate Forward Model
# -------------------------------------------------------------------
# samples = dict(mcmc.get_samples())
# For a marginalized fit, the fluences are not sampled: add the post-hoc draws first, flattened
# to (draws, n_pulses) to match get_samples(), e.g.
#     samples["A"] = extra["A"].reshape(-1, extra["A"].shape[-1])
if "A" not in samples:
    raise KeyError("samples has no 'A'. For a marginalized fit, add the fluences drawn by "
                   "posthoc_samples first (see the comment above).")

num_draws = len(next(iter(samples.values())))
n_pulses = samples["mean_arrival_time"].shape[-1]

# Pick a draw index using JAX random
# key = jax.random.PRNGKey(110)
# key = jax.random.PRNGKey(5)

keep_chains = [1,5,14,25]                                        # the chains to draw from
n_per_chain = num_draws // mcmc.num_chains
allowed = np.concatenate([np.arange(c * n_per_chain, (c + 1) * n_per_chain) for c in keep_chains])
draw_idx = int(allowed[jax.random.randint(key, shape=(), minval=0, maxval=len(allowed))])

# Parameters of the chosen draw, each an array of shape (n_pulses,). Only the parameters the
# model reads are taken; 'A' is the fluence, which single_pulse_fn applies itself.
param_names = [k for k in ("mean_arrival_time", "sigma", "DM", "tau_ref", "c1", "c2",
                           "gamma", "beta", "delta", "A", "A_snr") if k in samples]
test_params = {k: jnp.asarray(samples[k][draw_idx]).reshape(n_pulses) for k in param_names}

print(f"  from chain {draw_idx // n_per_chain}")
print(f"Plotting for Posterior Draw #{draw_idx}, {n_pulses} pulse(s)")
print("A_snr",test_params["A_snr"])
print("sigma",test_params["sigma"])
print("tau_ref",test_params["tau_ref"])

# Evaluate every pulse (vmap over the pulse axis), then sum them for the full model
M_FD_pulses = jax.vmap(single_pulse_fn)(test_params)       # (n_pulses, nfreq, nfourier)
M_FD = jnp.sum(M_FD_pulses, axis=0)                        # (nfreq, nfourier)

# The data and model live on the retained Fourier bins only, so each bin is put back at its own
# index on the full rfft grid before inverting; irfft on the truncated array would reinterpret
# bin k as frequency k / (Nseg dt), which is not where it came from.
Nseg = burst.n_time
dt = burst.dt
n_full = Nseg // 2 + 1
bin_index = jnp.rint(jnp.asarray(burst.fourier_freqs).reshape(-1) * Nseg * dt).astype(int)
pad = lambda X: jnp.zeros(X.shape[:-1] + (n_full,), X.dtype).at[..., bin_index].set(X)

to_TD = lambda X: (1.0 / dt) * jnp.fft.irfft(pad(X), n=Nseg, axis=-1)
M_TD = to_TD(M_FD)
M_TD_pulses = to_TD(M_FD_pulses)                           # (n_pulses, nfreq, ntime)
data_TD = to_TD(burst.data_fd)

data_TD_plot = data_TD
model_TD_plot = M_TD
time_plot = burst.times
freq_plot = jnp.squeeze(burst.freq_channels)

# -------------------------------------------------------------------
# 2. Residuals & Frequency Collapsed Time Series
# -------------------------------------------------------------------
resid_TD_plot = data_TD_plot - model_TD_plot

# Frequency-summed time series
data_ts = jnp.nansum(data_TD_plot, axis=0)
model_ts = jnp.nansum(model_TD_plot, axis=0)
pulse_ts = jnp.nansum(M_TD_pulses, axis=1)                 # (n_pulses, ntime)
resid_ts = data_ts - model_ts

# Shared robust color limits using jnp
vmin, vmax = jnp.nanpercentile(data_TD_plot, jnp.array([1.0, 99.0]))

# -------------------------------------------------------------------
# 3. Figure Layout & Plotting
# -------------------------------------------------------------------
fig = plt.figure(figsize=(12, 6))
gs = gridspec.GridSpec(
    2,
    3,
    height_ratios=[1, 3],
    hspace=0.05,
    wspace=0.08,
    figure=fig,
)

ax_ts_data = fig.add_subplot(gs[0, 0])
ax_ts_model = fig.add_subplot(gs[0, 1], sharey=ax_ts_data)
ax_ts_resid = fig.add_subplot(gs[0, 2], sharey=ax_ts_data)

ax_wf_data = fig.add_subplot(gs[1, 0], sharex=ax_ts_data)
ax_wf_model = fig.add_subplot(gs[1, 1], sharex=ax_ts_model, sharey=ax_wf_data)
ax_wf_resid = fig.add_subplot(gs[1, 2], sharex=ax_ts_resid, sharey=ax_wf_data)

# --- Top row: Frequency-summed time series ---
ax_ts_data.plot(time_plot, data_ts, color="k", lw=0.8)
ax_ts_model.plot(time_plot, model_ts, color="k", lw=0.8, label="sum")
if n_pulses > 1:
    # Each component separately, and its arrival time (at the reference frequency)
    colors = plt.cm.tab10(np.arange(n_pulses) % 10)
    t0 = np.asarray(test_params["mean_arrival_time"])
    for j in range(n_pulses):
        ax_ts_model.plot(time_plot, pulse_ts[j], color=colors[j], lw=0.8, ls="--", label=f"pulse {j}")
        # for ax in (ax_ts_model, ax_wf_model):
        #     ax.axvline(t0[j], color=colors[j], lw=0.6, alpha=0.6)
    ax_ts_model.legend(fontsize=7, loc="upper right")
ax_ts_resid.plot(time_plot, resid_ts, color="k", lw=0.8)
ax_ts_resid.axhline(0, color="gray", lw=0.6, ls="--")

# --- Bottom row: Waterfalls ---
imshow_kwargs = dict(
    aspect="auto",
    origin="lower",
    interpolation="nearest",
    cmap="viridis",
    vmin=vmin,
    vmax=vmax,
    extent=(time_plot[0], time_plot[-1], freq_plot[0], freq_plot[-1]),
)

ax_wf_data.imshow(data_TD_plot, **imshow_kwargs)
ax_wf_model.imshow(model_TD_plot, **imshow_kwargs)
ax_wf_resid.imshow(resid_TD_plot, **imshow_kwargs)

# --- Formatting ---
ax_ts_data.set_title("Data")
ax_ts_model.set_title(f"Model (Draw {draw_idx}, {n_pulses} pulse{'s' if n_pulses > 1 else ''})")
ax_ts_resid.set_title("Residual")

ax_wf_data.set_xlabel("Time [s]")
ax_wf_model.set_xlabel("Time [s]")
ax_wf_resid.set_xlabel("Time [s]")
ax_wf_data.set_ylabel("Frequency [MHz]")

for ax in (ax_ts_data, ax_ts_model, ax_ts_resid):
    ax.tick_params(labelbottom=False)
for ax in (ax_ts_model, ax_wf_model, ax_ts_resid, ax_wf_resid):
    ax.tick_params(labelleft=False)

plt.tight_layout()
plt.show()

In [ ]:
s = {k: np.asarray(v) for k, v in mcmc.get_samples().items()}
j, k = np.triu_indices(s["A_snr"].shape[1], 1)
main = int(np.argmax(np.median(s["A_snr"], axis=0)))
pair = np.where(((j == main) & (k == 4)) | ((j == 4) & (k == main)))[0][0]
print(np.percentile(s["toa_separation"][:, pair], [5, 50, 95]))   # use draws from one chain (labels can swap)

In [ ]:
# -------------------------------------------------------------------
# 1. Forward model evaluation, Fourier domain
# -------------------------------------------------------------------
# test_params comes from the posterior-draw cell: each entry has shape (n_pulses,). Every pulse
# is evaluated (vmap over the pulse axis) and the pulses are summed into the full model.
M_FD = jnp.sum(jax.vmap(single_pulse_fn)(test_params), axis=0)     # (nfreq, nfourier)

# The data and the model already live on the truncated Fourier axis, which is exactly the set
# of bins the likelihood uses, so nothing needs slicing here.
data_reshaped_FFT = burst.data_fd
sigma_bins = jnp.broadcast_to(burst.sigma_noise_fd, data_reshaped_FFT.shape)

# Whitened, so all three panels are in units of sigma and the residual should be N(0, 1) noise
data_FD_plot = (data_reshaped_FFT / sigma_bins).real
model_FD_plot = (M_FD / sigma_bins).real
resid_FD_plot = data_FD_plot - model_FD_plot

fourier_freqs = burst.fourier_freqs
fourier_plot = jnp.asarray(fourier_freqs).reshape(-1)
freq_plot = jnp.squeeze(burst.freq_channels)

print("Max Data abs :", jnp.max(jnp.abs(data_FD_plot)))
print("Max Model abs:", jnp.max(jnp.abs(model_FD_plot)))
print(f"whitened residual sd: {float(jnp.std(resid_FD_plot)):.3f}  (expect 1)")

# -------------------------------------------------------------------
# 2. Channel-summed spectra
# -------------------------------------------------------------------
data_ts = jnp.nansum(data_FD_plot, axis=0)
model_ts = jnp.nansum(model_FD_plot, axis=0)
resid_ts = data_ts - model_ts

# Data & Model limits
vmin, vmax = jnp.nanpercentile(data_FD_plot, jnp.array([1.0, 99.0]))

# Independent zero-centered limits for Residuals
vmax_res = jnp.nanpercentile(jnp.abs(resid_FD_plot), 99.0)

# -------------------------------------------------------------------
# 3. Figure Layout & Plotting
# -------------------------------------------------------------------
fig = plt.figure(figsize=(12, 6))
gs = gridspec.GridSpec(
    2,
    3,
    height_ratios=[1, 3],
    hspace=0.05,
    wspace=0.08,
    figure=fig,
)

ax_ts_data = fig.add_subplot(gs[0, 0])
ax_ts_model = fig.add_subplot(gs[0, 1], sharey=ax_ts_data)
ax_ts_resid = fig.add_subplot(gs[0, 2], sharey=ax_ts_data)

ax_wf_data = fig.add_subplot(gs[1, 0], sharex=ax_ts_data)
ax_wf_model = fig.add_subplot(gs[1, 1], sharex=ax_ts_model, sharey=ax_wf_data)
ax_wf_resid = fig.add_subplot(gs[1, 2], sharex=ax_ts_resid, sharey=ax_wf_data)

# --- Top row: channel-summed spectra ---
ax_ts_data.plot(fourier_plot, data_ts, color="k", lw=0.8)
ax_ts_model.plot(fourier_plot, model_ts, color="k", lw=0.8)
ax_ts_resid.plot(fourier_plot, resid_ts, color="k", lw=0.8)
ax_ts_resid.axhline(0, color="gray", lw=0.6, ls="--")

# --- Bottom row: Waterfalls ---
imshow_data_kwargs = dict(
    aspect="auto",
    origin="lower",
    interpolation="nearest",
    cmap="viridis",
    vmin=vmin,
    vmax=vmax,
    extent=(fourier_plot[0], fourier_plot[-1], freq_plot[0], freq_plot[-1]),
)

imshow_resid_kwargs = dict(
    aspect="auto",
    origin="lower",
    interpolation="nearest",
    cmap="viridis",
    vmin=-vmax_res,
    vmax=vmax_res,
    extent=(fourier_plot[0], fourier_plot[-1], freq_plot[0], freq_plot[-1]),
)

ax_wf_data.imshow(data_FD_plot, **imshow_data_kwargs)
ax_wf_model.imshow(model_FD_plot, **imshow_data_kwargs)
ax_wf_resid.imshow(resid_FD_plot, **imshow_resid_kwargs)

# --- Formatting ---
ax_ts_model.set_title(f"Re(Model) (Draw {draw_idx})")
ax_ts_data.set_title("Re(Data)")
ax_ts_resid.set_title("Re(Residuals)")

ax_wf_data.set_xlabel("Fourier frequency [Hz]")
ax_wf_model.set_xlabel("Fourier frequency [Hz]")
ax_wf_resid.set_xlabel("Fourier frequency [Hz]")
ax_wf_data.set_ylabel("Frequency [MHz]")

for ax in (ax_ts_data, ax_ts_model, ax_ts_resid):
    ax.tick_params(labelbottom=False)
for ax in (ax_ts_model, ax_wf_model, ax_ts_resid, ax_wf_resid):
    ax.tick_params(labelleft=False)

plt.tight_layout()
plt.show()

In [ ]:
n_per_chain = len(samples["mean_arrival_time"]) // mcmc.num_chains

keep_chains = range(mcmc.num_chains)        # or the chains you kept, e.g. [11, 14, 15]
allowed = np.concatenate([np.arange(c * n_per_chain, (c + 1) * n_per_chain) for c in keep_chains])

# A typical draw: random, from the kept chains
draw_idx = int(np.random.default_rng().choice(allowed))
# ...or the single most probable draw (lowest potential energy):
# pe = np.asarray(mcmc.get_extra_fields()["potential_energy"])
# draw_idx = int(allowed[np.argmin(pe[allowed])])
print(f"draw {draw_idx}, chain {draw_idx // n_per_chain}")

M_FD, M_FD_pulses = model_from_draw(single_pulse_fn, samples, draw_idx)

plot_fit_TD(burst, M_FD, M_FD_pulses, title_model=f"Model (draw {draw_idx})");
plot_fit_FD(burst, M_FD, title_model=f"Re(Model) (draw {draw_idx})");